## Initializing the functions:

In [0]:
%run ../../Configurations/Init_Scripts 

## Declaring the source and destination paths:

In [0]:
dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")
 
dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
 
dbutils.widgets.text('ExternalLocation_silver',"/mntprod_silver")
ExternalLocation_silver = dbutils.widgets.get('ExternalLocation_silver')

In [0]:
# Source Path:
#-----srcfile_BW_equipmaster = '/mntprod_raw/SAP/S4/EquipmentMaster'
srcfile_BW_equipmaster = ExternalLocation_raw+'/SAP/S4/EquipmentMaster'
# Source path for Product Hierarchy:
#------src_prodhrchy = '/mnt/silver/DIMProductHierarchy'
src_prodhrchy = ExternalLocation_silver+'/DIMProductHierarchy'
# destination folder for EquipmentMasterCS_Historical:
#------dst_EquipmentMaster = '/mnt/silver/DIMEquipmentMaster'
dst_EquipmentMaster = ExternalLocation_silver+'/DIMEquipmentMaster'

checkPointLocation = dst_EquipmentMaster+'/_checkpoints/'
configId = 15
sourceTypeId = 2
queueName = 'sapdataprocess-queue'
CreatedBy = 'ADB_EquipmentMaster'
DeviceTypeCd = 'SAP'
MessageTypeCd = 'EquipmentMaster'

RefGoods = ['-U','-R','_R','_U']

In [0]:
subscriptionId = dbutils.secrets.get("ABV_AKV_ADB_SCOPE","ABV-SubscriptionId")
tenantId = dbutils.secrets.get("ABV_AKV_ADB_SCOPE","ABV-TenantId")
clientId = dbutils.secrets.get("ABV_AKV_ADB_SCOPE","ABV-agnadb-SPN-ClientId")
clientSecret = dbutils.secrets.get("ABV_AKV_ADB_SCOPE","ABV-agnadb-SPN-Secret")
resourceGroup = dbutils.secrets.get("ABV_AKV_ADB_SCOPE","ABV-ResourceGroup")
queueConnectionString = dbutils.secrets.get("ABV_AKV_ADB_SCOPE","ABV-QueueConnectionString")

## Data Ingestion

In [0]:
from azure.servicebus import ServiceBusClient, ServiceBusMessage, AutoLockRenewer
from datetime import datetime
import json

In [0]:
dbutils.widgets.text("ingestionQueueName", "sapingestionframeworkequipmentmasterqueue")
ingestionQueueName = dbutils.widgets.get("ingestionQueueName")

dbutils.widgets.text('IngestionConfigId','5')
IngestionConfigId = dbutils.widgets.get('IngestionConfigId')

dbutils.widgets.text('SourceTypeId','2')
SourceTypeId = dbutils.widgets.get('SourceTypeId')

job_id=dbutils.widgets.text("job_id","-1")
job_id=dbutils.widgets.get("job_id")

run_id=dbutils.widgets.text("run_id","-1")
run_id=dbutils.widgets.get("run_id")

serviceBusConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="CSServiceBus")

#---dbutils.widgets.text('SourceFilePathBW','/mnt/raw/SAP/BW/EquipmentMaster/')
dbutils.widgets.text('SourceFilePathS4',ExternalLocation_raw+'/SAP/S4/EquipmentMaster/')
DestFilePathBW = dbutils.widgets.get('SourceFilePathBW')

#---dbutils.widgets.text('SourceFilePathS4','/mnt/raw/SAP/S4/EquipmentMaster/')
dbutils.widgets.text('SourceFilePathS4',ExternalLocation_raw+'/SAP/S4/EquipmentMaster/')
DestFilePathS4 = dbutils.widgets.get('SourceFilePathS4')

In [0]:
def MessageProcessing(message,receiver):
    try:
        print("Current Time = {}".format(datetime.now()))
        message_json = json.loads(str(message))
        subject = message_json.get('subject')

        sourceFilePath = subject.replace('/blobServices/default/containers','').replace('/blobs','')
        sourceFilePathList = sourceFilePath.split('/')

        sourceContainerPath = sourceFilePathList[1]
        sourceFolderPath = sourceFilePath[sourceFilePath.find('/',1)+1:sourceFilePath.rfind('/',1)]
        sourceFileName = sourceFilePathList[-1]
        if sourceFileName.startswith('ZBWI05226_AGN_CON'):
            destFilePath = DestFilePathS4
            destinationFolderPath = destFilePath[destFilePath.find('/',1)+1:destFilePath.rfind('/',1)]
            print(destFilePath)
        elif sourceFileName.startswith('ZLTQ_EQUIPMENT_MASTER'):
            destFilePath = DestFilePathBW
            destinationFolderPath = destFilePath[destFilePath.find('/',1)+1:destFilePath.rfind('/',1)]
            print(destFilePath)

        dbutils.fs.cp(f"/{ExternalLocation_raw}/"+sourceFilePath,destFilePath)
        receiver.complete_message(message)
        print("File Copied")
        processedFileList =([{'ConfigId':IngestionConfigId,'SourceTypeId':SourceTypeId,'SourceContainerPath':sourceContainerPath,'SourceFolderPath':sourceFolderPath,'SourceFileName':sourceFileName,
        'DestinationContainerPath':"raw",'DestinationFolderPath':destinationFolderPath,'DestinationFileName':sourceFileName,'PipelineStatus':"Succeeded",'PipelineRunId':run_id, 'JobId':job_id,'DeviceType':DeviceTypeCd,'LogType':MessageTypeCd
        }]) 
        logIntoIngestionLogTable(processedFileList,'LoadFiles_EquipmentMaster')
    except Exception as e:
        (receiver.dead_letter_message(message,reason="Failed"))
        processedFileList =([{'ConfigId':IngestionConfigId,'SourceTypeId':SourceTypeId,'SourceContainerPath':sourceContainerPath,'SourceFolderPath':sourceFolderPath,'SourceFileName':sourceFileName,'DestinationContainerPath':"raw",'DestinationFolderPath':destinationFolderPath,'DestinationFileName':sourceFileName,'PipelineStatus':"Failed",'PipelineRunId':run_id, 'JobId':job_id,'DeviceType':DeviceTypeCd,'LogType':MessageTypeCd,'ErrorMessage':str(e)
        }]) 
        logIntoIngestionLogTable(processedFileList,'LoadFiles_EquipmentMaster') 
        raise e

In [0]:
try:
    with ServiceBusClient.from_connection_string(serviceBusConnectionString) as sb_client:
        with sb_client.get_queue_receiver(ingestionQueueName) as receiver:
            renewer = AutoLockRenewer()
            while True:
                receivedMessages = receiver.receive_messages(max_message_count=20,max_wait_time=5)
                if(len(receivedMessages)>0):
                    for message in receivedMessages:
                        renewer.register(receiver, message, max_lock_renewal_duration=6000)
                        MessageProcessing(message,receiver)
                else:
                    print("No Messages in the queue")
                    break
except Exception as e:
    raise e

## Defining Schema:

In [0]:
# Defining Schema:
Schema_BW_equip = StructType([
StructField("EquipmentKey", StringType(), False),
StructField("ManSerialNumberNbr", StringType(), False),
StructField("SerialNumberNbr", StringType(), False),
StructField("Mfg_Batch", StringType(), False),
StructField("ProductCd", StringType(), False),
StructField("CreatedOnDtKey", StringType(), False),
StructField("ChangedByNm", StringType(), False),
StructField("ChangedOnDtKey", StringType(), False),
StructField("CreatedByNm", StringType(), False),
StructField("ShipToID", StringType(), False),
StructField("SoldToID", StringType(), False),
StructField("EquipmentTransitStatusNm", StringType(), False),
StructField("WarrantyEndDt", StringType(), False),
StructField("WarrantyStartDt", StringType(), False),
StructField("StockTypeInd", StringType(), False),
StructField("ZINBDt", StringType(), False),
StructField("EquipmentCategoryCd", StringType(), False),
StructField("SalesOrgCd", StringType(), False),
StructField("UpdateModeCd", StringType(), False),
StructField("StatusUpdateCd", StringType(), False)])

In [0]:
# bwqueueName = 'sapbwequipmentdataprocess-queue'
df_src_BW_EquipMaster = (spark.readStream.format("cloudFiles")
                              .option("cloudFiles.subscriptionId", subscriptionId)
                              .option("cloudFiles.connectionString", queueConnectionString)
                              .option("cloudFiles.queueName", queueName)
                              .option("cloudFiles.resourceGroup", resourceGroup)
                              .option("cloudFiles.tenantId", tenantId)
                              .option("cloudFiles.clientId", clientId)
                              .option("cloudFiles.clientSecret", clientSecret)
                              .option("cloudFiles.allowOverwrites","true")
                              .option("cloudFiles.includeExistingFiles","false")
                              .option("cloudFiles.backfillInterval",'1 day')             
                              .option("cloudFiles.useNotifications",'true')
                              .option("cloudFiles.format", "csv")
                              .option("rescuedDataColumn", "_rescued_data") # makes sure that you don't lose data
                              .option("delimiter", "|")
                              .schema(Schema_BW_equip) # provide a schema here for the files
                              .load(srcfile_BW_equipmaster)
                              .select("*",col("_metadata.file_path").alias('SourceFilePath'),
                                          col("_metadata.file_name").alias('SourceFileName'),
                                          col("_metadata.file_modification_time").alias('LoadDateTmstmp'),
                                          col("_metadata.file_size").alias('SourceFileSize')
                                     )
                              .withColumn('SourceFilePath', regexp_replace(regexp_replace('SourceFilePath','dbfs:/mnt/raw/',''),'/mnt/raw/',''))
                              .withColumn('ConfigId', lit(configId).cast('int'))
                              .withColumn('SourceTypeId', lit(sourceTypeId).cast('int'))
                              .withColumn("DeviceType",lit(DeviceTypeCd))
                              .withColumn("LogType",lit(MessageTypeCd))
                              .withColumn("DeviceId",lit(None).cast('string'))
                              .filter((col("StatusUpdateCd") != 'SCRP'))
                    )

In [0]:
# Read Product Hierarchy from silver zone:
df_ProdHrchy = (spark.read.format('delta')
                          .load(src_prodhrchy)
                          .select('ProductCd', 'ProductDesc')
                          .withColumnRenamed('ProductCd','MaterialCd')                
                          .withColumnRenamed('ProductDesc','L2ProductLineNm')         
                          .dropDuplicates()
               )

# Read logsourcefileprocessed from silver zone:
W_LatestFile = (Window.partitionBy(col('ConfigId'),col('SourceFilePath'),col('LogFileStatus')).orderBy(col('UpdatedDt').desc()))
df_Logsourcefileprocessed = (spark.read.format('delta').load(src_filesProcessed)
                            #  .filter(col('ConfigId') == configId)
                             .withColumn('rownum',row_number().over(W_LatestFile))
                             .filter((col('LogFileStatus') == 'InProgress') & (col('ConfigId') == configId) & (col('rownum')==1))
                             .select('SourceFilePath','FileNameUUID')
                            )

df_EquipmentMaster_Historical = (spark.read.format('delta').load(dst_EquipmentMaster)
                                .drop('ShipEndDt','InventoryMovementStatusDesc','ReturnDt')
                                .dropDuplicates()
                     )

In [0]:
def upsertToDelta(df_Source, batchId):
    try:
        df_Source.persist()
        # loadAuditTables_DIM_InProgress(df_Source,dst_EquipmentMaster,configId,sourceTypeId,'ADB_EquipmentMaster')
        loadAuditTables_Ingestion_Log(df_Source,dst_EquipmentMaster,CreatedBy,'InProgress','')
        df_Sources = (df_Source.groupBy("SourceFilePath","SourceFileName","SourceFileSize","ConfigId","SourceTypeId",'DeviceId').count()
                          .withColumnRenamed("count",'RecdCnt')
                          .withColumn("FileNameDeviceTypeCd",lit(DeviceTypeCd))
                          .withColumn("ExternalSerialNbr",lit(None))
                          .withColumn("InternalSerialNbr",lit(None))
                          .withColumn("FileNameMessageTypeCd",lit(MessageTypeCd))
                          .withColumn("FileNameDtTmstmp",
                          regexp_replace(regexp_replace(
                            when(col('SourceFileName').like('ZLTQ_%'),concat(split(col('SourceFilePath'),'_').getItem(3),split(col('SourceFilePath'),'_').getItem(4)))
                            .otherwise(split(col('SourceFilePath'),'_').getItem(4)),'.CSV',''),'.csv',''))
                          .withColumn("FileNameApplicatorPortCd",lit(None))
                          .withColumn("FileNameCycleNbr",lit(None))
                          .withColumn('LogStartDate',lit(None))
                          .withColumn('LogEndDate',lit(None))
                          )

        loadlogProcessesDeltaTable(df_Sources,dst_EquipmentMaster,CreatedBy,'InProgress','')

        df_Source_Processed = (df_Source
                    .withColumn('EquipmentMasterUUID',expr('uuid()'))
                    .withColumn('ChangedOnDateKey', when((col('ChangedOnDtKey') == '00000000') | (col('ChangedOnDtKey').isNull()) | (col('ChangedOnDtKey') == '19000101'),col('CreatedOnDtKey'))
                                                         .otherwise(col('ChangedOnDtKey')))
                    .withColumn('ChangedByNm', when(col('ChangedByNm').isNull(),'ADB_EquipmentMaster')
                                                        .otherwise(col('ChangedByNm')))
                    .withColumn('ShipStartDt', to_date(col('ChangedOnDateKey'),'yyyyMMdd'))
                    .withColumn('SerialNumberNbr',trim(upper(col('SerialNumberNbr'))))
                    .withColumn('FilePriority', when(split(col('SourceFilePath'),'/').getItem(1) == 'BW','1')
                                               .when(split(col('SourceFilePath'),'/').getItem(1) == 'S4','2')
                                               .otherwise(""))
                    .withColumn('IsRefGood',when(reverse(substring(reverse(col('ProductCd')),1,2)).isin(RefGoods),1)
                                            .otherwise(0))
                    #.withColumn('MaterialCdRef',when(col('IsRefGood') == 1,reverse(substring(reverse(col('ProductCd')),3,100))).otherwise(col('ProductCd')))
                    .withColumn('CreatedBy',lit("ADB_EquipmentMaster"))
                    .withColumn('CreatedDt',lit(current_timestamp()))
                    .withColumn('UpdatedBy',lit("ADB_EquipmentMaster"))
                    .withColumn('UpdatedDt',lit(current_timestamp()))
                    .withColumnRenamed('CreatedOnDtKey', 'CreatedOnDateKey')
                    .withColumn('Mfg_BatchNbr', trim(upper(col('Mfg_Batch'))))
                    .withColumnRenamed('ProductCd', 'MaterialCd')
                    .withColumnRenamed('SalesOrgCd', 'SalesOrgNbr')
                    .withColumnRenamed('UpdateModeCd', 'UndefinedCd')
                    .withColumnRenamed('StatusUpdateCd', 'Equipment_Status')
                    .drop('ChangedOnDtKey','CreatedOnDtKey')
                   )
        
        df_Source_Prod = (df_Source_Processed.join(df_Logsourcefileprocessed,'SourceFilePath',how = 'inner')
                                             .join(df_ProdHrchy, 'MaterialCd',how = 'inner'))
        
        '''
        #Getting distinct record, based on refurbished good
        WRefGoods = (Window.partitionBy(col('EquipmentTransitStatusNm'),col('SerialNumberNbr'),col('ChangedOnDateKey'),col('FileNameUUID'),col('ShipToID'),col('SoldToID'),col('MaterialCdRef')).orderBy(col('IsRefGood').desc()))
        df_Source_Prod = df_Source_Prod.withColumn("rk",row_number().over(WRefGoods)).filter("rk = 1")
        '''
        # assert df_Source_Prod.count() > 0, "Dataset join with logsourcefileprocessed and Product Hiearchy resulted Empty"
        if (df_Source_Prod.count() == 0):
            return "Dataset join with logsourcefileprocessed and Product Hiearchy resulted Empty"
        
        df_EquipmentMaster_History = df_EquipmentMaster_Historical.alias('src').join(df_Source_Prod.alias('dest'),'SerialNumberNbr','inner').select('src.*')
        
        df_EquipmentMaster_Delta = df_EquipmentMaster_History.unionByName(df_Source_Prod,allowMissingColumns=True)

        #If file is being reprocessed or same record being processed multiple times then should not create multiple rows with same shipStart AND ShipEnd        
        W_IdentifyDuplicates = (Window.partitionBy(col('EquipmentKey'),col('SerialNumberNbr'),col('ChangedOnDateKey')).orderBy(col('CreatedDt').desc()))
        df_EquipmentMaster_Delta_dropDuplicates = df_EquipmentMaster_Delta.withColumn('SerialNumberNbr',when(col('SerialNumberNbr').isNull(),'UNKNOWN').otherwise(col('SerialNumberNbr'))).withColumn('DuplicateRow',row_number().over(W_IdentifyDuplicates)).filter('DuplicateRow == 1').drop('DuplicateRow')

        W_LoadOrderNumber = (Window.partitionBy(col('SerialNumberNbr')).orderBy(col('ChangedOnDateKey').desc(),col("FilePriority").desc(),col("LoadDateOrderNbr").asc(),col('LoadDateTmstmp').desc(),col('IsRefGood').desc()))
        df_EquipMaster_ShipDates = (df_EquipmentMaster_Delta_dropDuplicates
                                              .withColumn('EquipmentTransitStatusNm_Lead',lead('EquipmentTransitStatusNm',1).over(W_LoadOrderNumber))      
                                              .withColumn('ShipStartDt_Lag', lag('ShipStartDt',1).over(W_LoadOrderNumber))
                                              #.withColumn('ShipToID_Lead',lead('ShipToID',1).over(W_LoadOrderNumber))
                                              .withColumn('ShipEndDt', when(col('ShipStartDt_Lag').isNull(),'2111-11-11').otherwise(col('ShipStartDt_Lag')))
                                              .withColumn('LoadDateOrderNbr',row_number().over(W_LoadOrderNumber))
                                              .withColumn('InventoryMovementStatusDesc', 
                                                          when((col('EquipmentTransitStatusNm_Lead') == 'SHIPPED') & (col('EquipmentTransitStatusNm') == 'IN WAREHOUSE'), 
                                                               'RETURNED').otherwise(col('EquipmentTransitStatusNm')))
                                              .withColumn('ReturnDt',when((col('InventoryMovementStatusDesc') == 'RETURNED'), col('ShipStartDt')).otherwise(None))
                                              #.withColumn('WarehouseCode',when((col('InventoryMovementStatusDesc') == 'RETURNED'), col('ShipToID_Lead')).otherwise(None))
                                              .drop('EquipmentTransitStatusNm_Lead','ShipStartDt_Lag')
                                             )
        
        DL_EquipMaster_Hist = DeltaTable.forPath(spark, dst_EquipmentMaster)  
        (DL_EquipMaster_Hist.alias("tgt")
                .merge(
                df_EquipMaster_ShipDates.alias("src"),
                    "tgt.EquipmentKey = src.EquipmentKey and tgt.SerialNumberNbr = src.SerialNumberNbr and tgt.ChangedOnDateKey = src.ChangedOnDateKey")
          .whenNotMatchedInsert(values =
          {
            "tgt.EquipmentMasterUUID" : "src.EquipmentMasterUUID",
            "tgt.FileNameUUID" : "src.FileNameUUID",
            "tgt.EquipmentKey" : "src.EquipmentKey",
            "tgt.ManSerialNumberNbr" : "src.ManSerialNumberNbr",
            "tgt.SerialNumberNbr" : "src.SerialNumberNbr",
            "tgt.Mfg_BatchNbr" : "src.Mfg_BatchNbr",
            "tgt.MaterialCd" : "src.MaterialCd",
            "tgt.L2ProductLineNm" : "src.L2ProductLineNm",
            "tgt.CreatedOnDateKey" : "src.CreatedOnDateKey",
            "tgt.CreatedByNm" : "src.CreatedByNm",
            "tgt.ChangedOnDateKey" : "src.ChangedOnDateKey",
            "tgt.ChangedByNm" : "src.ChangedByNm",
            "tgt.ShipStartDt" : "src.ShipStartDt",
            "tgt.ShipEndDt" : "src.ShipEndDt",
            "tgt.ReturnDt" : "src.ReturnDt",
            "tgt.ShipToID" : "src.ShipToID",
            "tgt.SoldToID" : "src.SoldToID",
            #"tgt.WarehouseCode" : "src.WarehouseCode",
            "tgt.EquipmentTransitStatusNm" : "src.EquipmentTransitStatusNm",
            "tgt.InventoryMovementStatusDesc" : "src.InventoryMovementStatusDesc",
            "tgt.WarrantyEndDt" : "src.WarrantyEndDt",
            "tgt.WarrantyStartDt" : "src.WarrantyStartDt",
            "tgt.StockTypeInd" : "src.StockTypeInd",
            "tgt.SalesOrgNbr" : "src.SalesOrgNbr",
            "tgt.UndefinedCd" : "src.UndefinedCd",
            "tgt.Equipment_Status" : "src.Equipment_Status",
            "tgt.LoadDateTmstmp" : "src.LoadDateTmstmp",
            "tgt.CreatedBy" : "src.CreatedBy",
            "tgt.CreatedDt" : "src.CreatedDt",
            "tgt.UpdatedBy" : "src.UpdatedBy",
            "tgt.UpdatedDt" : "src.UpdatedDt",
            "tgt.LoadDateOrderNbr" : "src.LoadDateOrderNbr"
          })      
         .whenMatchedUpdate(set =
          {
            "tgt.FileNameUUID" : "src.FileNameUUID",
            "tgt.EquipmentKey" : "src.EquipmentKey",
            "tgt.ManSerialNumberNbr" : "src.ManSerialNumberNbr",
            "tgt.SerialNumberNbr" : "src.SerialNumberNbr",
            "tgt.Mfg_BatchNbr" : "src.Mfg_BatchNbr",
            "tgt.MaterialCd" : "src.MaterialCd",
            "tgt.L2ProductLineNm" : "src.L2ProductLineNm",
            "tgt.CreatedOnDateKey" : "src.CreatedOnDateKey",
            "tgt.CreatedByNm" : "src.CreatedByNm",
            "tgt.ChangedOnDateKey" : "src.ChangedOnDateKey",
            "tgt.ChangedByNm" : "src.ChangedByNm",
            "tgt.ShipStartDt" : "src.ShipStartDt",
            "tgt.ShipEndDt" : "src.ShipEndDt",
            "tgt.ReturnDt" : "src.ReturnDt",
            "tgt.ShipToID" : "src.ShipToID",
            "tgt.SoldToID" : "src.SoldToID",
           #"tgt.WarehouseCode" : "src.WarehouseCode",
            "tgt.EquipmentTransitStatusNm" : "src.EquipmentTransitStatusNm",
            "tgt.InventoryMovementStatusDesc" : "src.InventoryMovementStatusDesc",
            "tgt.WarrantyEndDt" : "src.WarrantyEndDt",
            "tgt.WarrantyStartDt" : "src.WarrantyStartDt",
            "tgt.StockTypeInd" : "src.StockTypeInd",
            "tgt.SalesOrgNbr" : "src.SalesOrgNbr",
            "tgt.UndefinedCd" : "src.UndefinedCd",
            "tgt.Equipment_Status" : "src.Equipment_Status",
            "tgt.LoadDateTmstmp" : "src.LoadDateTmstmp",
            "tgt.CreatedBy" : "src.CreatedBy",
            "tgt.CreatedDt" : "src.CreatedDt",
            "tgt.UpdatedBy" : "src.UpdatedBy",
            "tgt.UpdatedDt" : current_timestamp(),
            "tgt.LoadDateOrderNbr" : "src.LoadDateOrderNbr"
          })
          .execute())

        # loadAuditTables_DIM_Complete(df_Source,dst_EquipmentMaster,configId,sourceTypeId,'ADB_EquipmentMaster','Succeeded','')
        df_Source1 = (df_EquipMaster_ShipDates.withColumn('ConfigId',lit(configId)).groupBy('SourceFilePath','SourceFileName','FileNameUUID','ConfigId')
                                    .agg(min(col('LoadDateTmstmp')).alias('LogStartDate'),
                                         max(col('LoadDateTmstmp')).alias('LogEndDate')))
        loadlogProcessesDeltaTable(df_Source1,dst_EquipmentMaster,CreatedBy,'Succeeded','')
        loadAuditTables_Ingestion_Log(df_Source,dst_EquipmentMaster,CreatedBy,'Succeeded','')
        df_Source.unpersist()
    except Exception as e:
        # loadAuditTables_DIM_Complete(df_Source,dst_EquipmentMaster,configId,sourceTypeId,'ADB_EquipmentMaster','Failed',str(e))
        loadlogProcessesDeltaTable(df_Source1,dst_EquipmentMaster,CreatedBy,'Failed',str(e))
        loadAuditTables_Ingestion_Log(df_Source,dst_EquipmentMaster,CreatedBy,'Failed',str(e))
        raise



In [0]:
# Reading the file and applying transformations:

(df_src_BW_EquipMaster.writeStream
                  .format("delta")
                  .trigger(availableNow=True)
                  .foreachBatch(upsertToDelta)
                  .option("checkpointLocation", checkPointLocation)
                  .outputMode("update")
                  .start()
                  .awaitTermination()
)
